# Customer Churn Prediction & Retention Analytics

**Customer Analytics | Predictive Analytics | Business Intelligence**

Portfolio project for Business Analytics, Analytics Consulting, Data Analytics, and Customer Analytics roles.

## Executive Summary

Customer churn is a direct threat to revenue stability, customer lifetime value, and long-term business growth. Retaining existing customers is often more cost-effective than acquiring new ones, which makes churn prediction a high-impact analytics use case for customer-facing businesses.

This notebook analyzes customer behavior and business attributes to identify churn drivers, segment high-risk customers, estimate value at risk, and build predictive models that support targeted retention decisions. The output is designed as a consulting-style analytics case: it connects data analysis to business action.

## Research Question

How can customer behavior and business data be used to predict customer churn and generate actionable retention strategies?

## Core Narrative

**Data -> Analysis -> Insight -> Business Recommendation**

## Business Objective

The objective is to support customer retention, revenue preservation, customer segmentation, and data-driven decision making. The analysis helps prioritize customers and segments where retention action can create the greatest business value.

## Business Context

Organizations care about churn because losing customers reduces recurring revenue, weakens customer lifetime value, and increases pressure on acquisition spending. A churn analytics program helps businesses detect early warning signals, understand the profile of at-risk customers, and design retention strategies before value is lost.

## Dataset Overview

The project uses a bank customer churn dataset selected for customer analytics and predictive modeling. The dataset contains **10,000 customer records** with demographic, geographic, behavioral, product, satisfaction, and financial attributes.

Important variables include credit score, geography, gender, age, tenure, account balance, number of products, active member status, estimated salary, complaint flag, satisfaction score, card type, and loyalty points.

The target variable is **Exited**, standardized in the analysis as **churn**.

## Key Business Questions

- Which customers are most likely to churn?
- What customer characteristics influence churn?
- Which customer segments contribute maximum revenue risk?
- Which factors have the highest predictive importance?
- How can businesses prioritize retention efforts?

## Project Workflow

```text
Dataset
  -> Data Understanding
  -> Data Cleaning
  -> Exploratory Data Analysis
  -> SQL Analysis
  -> Statistical Analysis
  -> Machine Learning
  -> Business Insights
  -> Recommendations
```

## Analytical Methodology

The analysis begins with data understanding and quality checks, then moves into cleaning, feature engineering, exploratory analysis, SQL-based business analysis, statistical testing, predictive modeling, and business recommendation development. Each stage is structured to translate technical findings into retention-focused business decisions.

## Technologies Used

- Python
- SQL
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-Learn
- XGBoost
- LightGBM
- Power BI

## Skills Demonstrated

- Business Analytics
- Customer Analytics
- Exploratory Data Analysis
- Statistical Analysis
- Predictive Modelling
- Machine Learning
- SQL Analytics
- Customer Segmentation
- Business Insight Generation
- Data Storytelling

# 01 Data Understanding

Customer Churn Prediction & Retention Analytics

This notebook follows the consulting analytics flow: data, analysis, insight, and recommendation. Run from the repository root.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

raw = pd.read_csv('../data/raw/Customer-Churn-Records.csv')
raw.head()

In [ ]:
raw.shape, raw.dtypes

In [ ]:
raw.isna().sum().sort_values(ascending=False)

In [ ]:
raw.duplicated().sum()

In [ ]:
raw.describe().T

## Churn Distribution

The dataset contains 10,000 customers and an observed churn rate of 20.4%. This creates a realistic class imbalance problem for retention analytics.

In [ ]:
raw['Exited'].value_counts(normalize=True).rename('share')

# 02 Data Cleaning

Customer Churn Prediction & Retention Analytics

Documents the feature engineering used in the processed dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

df = pd.read_csv('../data/processed/customer_churn_processed.csv')
df.head()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
df[['age_group','tenure_group','monthly_spend_category','customer_value_segment','annual_value_proxy','customer_lifetime_value','revenue_at_risk','high_risk_customer_flag']].head()

The bank dataset does not contain direct monthly spend. The project therefore defines a documented annual relationship value proxy from balance, estimated salary, product count, and loyalty points.

# 03 Exploratory Data Analysis

Customer Churn Prediction & Retention Analytics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

df = pd.read_csv('../data/processed/customer_churn_processed.csv')

In [ ]:
pd.crosstab(df['geography'], df['churn'], normalize='index')

### Observation
Germany has the highest churn rate in the portfolio.

### Business Interpretation
Regional service experience or product fit may be materially different.

### Recommendation
Prioritize Germany-specific save offers and root-cause interviews.

In [ ]:
sns.barplot(data=df.groupby('age_group', observed=False)['churn'].mean().reset_index(), x='age_group', y='churn')
plt.title('Churn Rate by Age Group')

### Observation
Middle-aged and older groups show elevated attrition.

### Business Interpretation
Financial needs may become more complex with age, making service gaps more visible.

### Recommendation
Create proactive relationship reviews for customers above the high-risk age threshold.

In [ ]:
sns.heatmap(df[['creditscore','age','tenure','balance','numofproducts','estimatedsalary','satisfaction_score','point_earned','churn']].corr(), annot=True, cmap='vlag', center=0)
plt.title('Correlation Matrix')

# 04 SQL Analysis

Customer Churn Prediction & Retention Analytics

This notebook uses SQLite to demonstrate the SQL logic from the `sql/` folder on the processed dataset.

In [ ]:
import pandas as pd, sqlite3
conn = sqlite3.connect(':memory:')
df = pd.read_csv('../data/processed/customer_churn_processed.csv')
df.to_sql('customer_churn', conn, index=False, if_exists='replace')

In [ ]:
pd.read_sql_query("""
SELECT gender, COUNT(*) AS customers, SUM(churn) AS churned, ROUND(100.0 * AVG(churn), 2) AS churn_rate_pct
FROM customer_churn
GROUP BY gender
ORDER BY churn_rate_pct DESC
""", conn)

In [ ]:
pd.read_sql_query("""
WITH segment_loss AS (
 SELECT customer_value_segment, SUM(revenue_at_risk) AS value_at_risk
 FROM customer_churn
 GROUP BY customer_value_segment
)
SELECT *, ROUND(100.0 * value_at_risk / SUM(value_at_risk) OVER (), 2) AS share_of_value_at_risk_pct
FROM segment_loss
ORDER BY value_at_risk DESC
""", conn)

# 05 Statistical Analysis

Customer Churn Prediction & Retention Analytics

Hypothesis tests quantify which observed differences are statistically meaningful.

In [ ]:
import pandas as pd
tests = pd.read_csv('../data/processed/statistical_tests.csv')
tests

Interpret tests with business judgment: statistical significance shows association, not automatic causality.

# 06 Machine Learning

Customer Churn Prediction & Retention Analytics

The modeling workflow compares Logistic Regression, Decision Tree, Random Forest, XGBoost, and LightGBM. The primary model excludes complaint and churn-derived fields to reduce leakage risk.

In [ ]:
import pandas as pd
metrics = pd.read_csv('../data/processed/model_metrics.csv')
metrics

In [ ]:
pd.read_csv('../data/processed/feature_importance.csv').head(15)

The tuned Random Forest achieved ROC-AUC of 0.868, giving a practical retention-ranking model for prioritizing outreach.